For each NACE Class get the 100 chunks that scored highest across all the reports 

In [14]:
import pandas as pd
import glob
import os
import tqdm

In [15]:
path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_3_stoxx/"
path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables_cos_sim_0.0_nace_level_1_stoxx/"

In [16]:
end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data", path.split("/")[-2])

In [17]:
reports = glob.glob(path + "*/*_short.csv")
len(reports)

71

In [18]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

In [19]:
# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):
    df = pd.read_csv(report)
    scores = df[[column for column in df.columns if "Scores" in column]].columns
    for score in scores: 
        temp = df[df[score].notna()][["Sentences", score]]
        temp["NACE_Code"] = score
        temp = temp.rename(columns={score: "Score"})
        result = pd.concat([result, temp])

  0%|          | 0/71 [00:00<?, ?it/s]/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_46869/964086622.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, temp])
100%|██████████| 71/71 [00:01<00:00, 41.50it/s]


In [20]:
os.makedirs(end_path, exist_ok=True)

In [21]:
recordings = []

In [22]:
# for each code, store the 100 with the highest similarity score to the code

for code in set(result["NACE_Code"].to_list()): 
    temp = result[result["NACE_Code"] == code] 
    temp = temp.drop_duplicates(subset="Sentences")
    temp = temp[temp["Sentences"].apply(len) >= 100]
    temp = temp.sort_values(by="Score", ascending=False)[:100]
    temp = temp[temp["Score"] >= 0.4]

    recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean()})

    text = ""
    for _,row in temp.iterrows():
        text += "Score: " + str(round(row["Score"], 2)) + "\n\n" + row["Sentences"] + "\n\n" 

    with open(os.path.join(end_path, code.replace("/"," ")) + ".txt", "w") as f:
        f.write(text)  
    

In [23]:
df_recordings = pd.DataFrame(recordings)
df_recordings.head()

,Code,Nbr. of Chunks,Avg. Length,Avg. Score
0,Scores_I_ACCOMMODATION AND FOOD SERVICE ACTIVI...,3,5367.666667,0.403012
1,"Scores_R_ARTS, ENTERTAINMENT AND RECREATION",1,300.000000,0.411062
2,Scores_J_INFORMATION AND COMMUNICATION,4,5782.000000,0.424055
3,Scores_Q_HUMAN HEALTH AND SOCIAL WORK ACTIVITIES,6,2546.000000,0.429892
4,Scores_H_TRANSPORTATION AND STORAGE,25,2769.200000,0.426653


In [24]:
df_recordings.to_csv(end_path + "/statistics.csv")

In [25]:
df_recordings["Nbr. of Chunks"].sum()

810

In [26]:
end_path

'/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/tables_cos_sim_0.0_nace_level_1_stoxx'